# Flatten the negotiated_rates.json file using Panda(Python library)

In [0]:


import pandas as pd
import json 

# 1. Load the raw data
with open('negotiated_rates.json') as f:
    data = json.load(f)


# 2. Flatten the first level
df = pd.json_normalize(
    data['out_of_network'],
    record_path=['allowed_amounts'],
    meta=['name', 'billing_code_type', 'billing_code', 'description'],
    errors='ignore'
) 


# 3. Explode the senond level
df = df.explode('payments.providers').reset_index(drop=True)


# 4. Flatten the dictionary inside the second level
providers_df = pd.json_normalize(df['payments.providers'])
df = pd.concat([df.drop(columns=['payments.providers']), providers_df], axis=1)

# 5. Explode the list inside the second level
df = df.explode('npi').reset_index(drop=True)

# 5. Clean up column names and select the requested fields
df = df.rename(columns={'payments.allowed_amount': 'allowed_amount'})
final_columns = [
    'name',
    'billing_code_type',
    'billing_code',
    'description',
    'service_code',
    'billing_class',
    'allowed_amount',
    'billed_charge',
    'npi'
]

# 6. Save the result
df_final = df[final_columns]

# 7. Display the result and save it as a csv file format
display(df_final)
df_final.to_csv('negotiated_rates.csv', index=False)


# Flatten the negotiated_rates.json file using Spark(Distributed System)

In [0]:


from pyspark.sql.functions import *
import json

# 1. Load the raw data
df = spark.read.option("multiline", "true").json("/Workspace/Users/hussainazimi.career@gmail.com/data-5035-2026/week03/negotiated_rates.json")


# 2. Explode the first level list into row
df_oon = df.select(explode(col("out_of_network")).alias("oon"))

# 3. Explode the allowed_amounts list inside each row
df_amounts = df_oon.select(
    col('oon.name').alias('name'),
    col('oon.billing_code_type').alias('billing_code_type'),
    col('oon.billing_code').alias('billing_code'),
    col('oon.description').alias('description'),
    explode(col('oon.allowed_amounts')).alias('allowed_amounts')
)

# 4. Flatten second level fields
df_providers = df_amounts.select(
    '*',
    col('allowed_amounts.service_code').alias('service_code'),
    col('allowed_amounts.billing_class').alias('billing_class'),
    col('allowed_amounts.payments.allowed_amount').alias('allowed_amount'),
    explode(col('allowed_amounts.payments.providers')).alias('providers')
)

# 5. Explode the list inside the second level and select the requested fields
df_final = df_providers.select(
    "name",
    "billing_code_type",
    "billing_code",
    "description",
    "service_code",
    "billing_class",
    "allowed_amount",
    col('providers.billed_charge').alias('billed_charge'),
    explode(col('providers.npi')).alias('npi')
    
)

# 8. Display the result
df_final.show()
